# Fastmagic Colab Benchmark Notebook

# AI-generated: Claude, 2026-04-23
This notebook is designed for rigorous IQL replication: preset hyperparameters, multi-seed sweeps, CSV aggregation, and backup of checkpoints/results to Google Drive.e.

In [ ]:
!nvidia-smi
!pip install -q torch==2.3.1 gym==0.24.1 d4rl==1.1 numpy==1.26.4 wandb==0.17.6 tqdm==4.66.5 pandas==2.2.2

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
!git clone https://github.com/<your-username>/fastmagic.git || true
%cd /content/fastmagic

In [ ]:
# AI-generated: Claude, 2026-04-23
from pathlib import Path

PRESET = 'mujoco'  # switch to 'antmaze' for AntMaze replication
SEEDS = [0, 1, 2]
MAX_ENVS = None  # set e.g. 1 for smoke tests
MIXED_PRECISION = True
PROFILE = False
DRIVE_ROOT = Path('/content/drive/MyDrive/fastmagic_benchmarks')

PRESET_ENVS = {
    'mujoco': [
        'halfcheetah-medium-v2', 'halfcheetah-medium-replay-v2', 'halfcheetah-medium-expert-v2',
        'hopper-medium-v2', 'hopper-medium-replay-v2', 'hopper-medium-expert-v2',
        'walker2d-medium-v2', 'walker2d-medium-replay-v2', 'walker2d-medium-expert-v2',
    ],
    'antmaze': [
        'antmaze-umaze-v2', 'antmaze-umaze-diverse-v2', 'antmaze-medium-play-v2',
        'antmaze-medium-diverse-v2', 'antmaze-large-play-v2', 'antmaze-large-diverse-v2',
    ],
}

SELECTED_ENVS = PRESET_ENVS[PRESET][:MAX_ENVS] if MAX_ENVS is not None else PRESET_ENVS[PRESET]
print({'preset': PRESET, 'envs': SELECTED_ENVS, 'seeds': SEEDS, 'mixed_precision': MIXED_PRECISION})

PRESET = 'mujoco'  # change to 'antmaze' for AntMaze replication
SEEDS = [0, 1, 2]
MAX_ENVS = None  # set to a small integer for a quick smoke test
MIXED_PRECISION = True
PROFILE = False
DRIVE_ROOT = Path('/content/drive/MyDrive/fastmagic_benchmarks')

PRESET_ENVS = {
    'mujoco': [
        'halfcheetah-medium-v2', 'halfcheetah-medium-replay-v2', 'halfcheetah-medium-expert-v2',
        'hopper-medium-v2', 'hopper-medium-replay-v2', 'hopper-medium-expert-v2',
        'walker2d-medium-v2', 'walker2d-medium-replay-v2', 'walker2d-medium-expert-v2',
    ],
    'antmaze': [
        'antmaze-umaze-v2', 'antmaze-umaze-diverse-v2', 'antmaze-medium-play-v2',
        'antmaze-medium-diverse-v2', 'antmaze-large-play-v2', 'antmaze-large-diverse-v2',
    ],
}

SELECTED_ENVS = PRESET_ENVS[PRESET][:MAX_ENVS] if MAX_ENVS is not None else PRESET_ENVS[PRESET]
print({'preset': PRESET, 'envs': SELECTED_ENVS, 'seeds': SEEDS, 'mixed_precision': MIXED_PRECISION})

In [ ]:
# AI-generated: Claude, 2026-04-23
import subprocess

for env_name in SELECTED_ENVS:
    print(f'Caching dataset: {env_name}')
    subprocess.run(['python', 'data/download_d4rl.py', '--env', env_name], check=True)

command = [
    'python', 'src/benchmark_iql.py',
    '--preset', PRESET,
    '--seeds', *[str(seed) for seed in SEEDS],
]

if MAX_ENVS is not None:
    command.extend(['--max_envs', str(MAX_ENVS)])
if MIXED_PRECISION:
    command.append('--mixed_precision')
if PROFILE:
    command.append('--profile')

print('Running benchmark command:', command)
subprocess.run(command, check=True)

In [ ]:
import shutil

benchmark_drive_dir = DRIVE_ROOT / PRESET
benchmark_drive_dir.mkdir(parents=True, exist_ok=True)

for source_dir in [Path('results/benchmarks'), Path('models/benchmarks')]:
    if source_dir.exists():
        destination = benchmark_drive_dir / source_dir.name
        if destination.exists():
            shutil.rmtree(destination)
        shutil.copytree(source_dir, destination)

print(f'Copied benchmark artifacts to {benchmark_drive_dir}')
raw_path = f'results/benchmarks/{PRESET}_raw_runs.csv'

aggregate_df = pd.read_csv(aggregate_path)
raw_df = pd.read_csv(raw_path)

display(aggregate_df)
display(raw_df.head())